# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
This notebook provides a walkthrough for loading and exploring the FAIR<sup>2</sup> colorectal cancer survivor dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")


## 2. Data Overview
Review available record sets, fields, and their IDs.

We will print the available record sets and their structure (fields and columns), always referencing by their `@id`s.

In [ ]:
# List all available record sets and their fields, columns, and IDs
record_set_list = list(dataset.record_sets)
print("Available record sets and their fields/columns by @id:")
for rs in record_set_list:
    print(f"\nRecordSet '@id': {rs.id}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field '@id': {field.id} (label: {getattr(field, 'label', field.id)})")
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for col in rs.columns:
            print(f"    Column '@id': {col.id} (label: {getattr(col, 'label', col.id)}, dtype: {getattr(col, 'data_type', None)})")

For demonstration, let's inspect the first available record set and preview its records. We'll use its `@id` throughout the notebook.

In [ ]:
# Print the first record set's @id and review its first few entries
if len(record_set_list) == 0:
    raise ValueError("No record sets were found in the dataset.")
primary_recordset = record_set_list[0]
primary_recordset_id = primary_recordset.id
print(f"\nUsing record set '@id': {primary_recordset_id}\n")
# Preview first 3 records (each is a dict with keys as field/column @id)
for i, rec in enumerate(dataset.records(record_set=primary_recordset_id)):
    if i >= 3:
        break
    pprint.pprint(rec)

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis. Each DataFrame will use the record set's `@id` as its key, and column names will be their `@id`s.

In [ ]:
dataframes = {}
for rs in record_set_list:
    rs_id = rs.id
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set '@id': {rs_id}, shape: {dataframes[rs_id].shape}")

# Show columns available in the primary record set
print(f"\nColumns in primary record set '@id': {primary_recordset_id}:\n{dataframes[primary_recordset_id].columns.tolist()}")
dataframes[primary_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps—filtering, normalizing, and grouping—on a numeric field by its `@id` and group by another field's `@id`. We'll use only `@id`s so results are unambiguous and reliable.

*Note: Adapt the numeric and group field `@id`s to align with your schema overview above.*

In [ ]:
# Choose an example numeric field and a grouping field by @id
df = dataframes[primary_recordset_id]
column_ids = df.columns.tolist()
# For demonstration, let's pick likely numeric and grouping fields from the column @ids
# (If unsure, update these to valid numeric/group fields found above)

# Example: Let's look for common clinical field @ids present in this data
candidate_numeric_ids = [col for col in column_ids if ('age' in col.lower()) or ('interval' in col.lower()) or ('count' in col.lower()) or ('years' in col.lower()) or ('number' in col.lower())]
# e.g., '@id': 'https://api.app.sen.science/frontiers/7862866/age_at_first_diagnosis' (update to match the real @id)
numeric_field_id = candidate_numeric_ids[0] if candidate_numeric_ids else column_ids[0]

# Similarly, for group field (e.g., sex, anatomical_site, msi_status)
potential_group_ids = [col for col in column_ids if (
    'sex' in col.lower() or 'site' in col.lower() or 'status' in col.lower() or 'location' in col.lower() or 'msi' in col.lower()
)]
group_field_id = potential_group_ids[0] if potential_group_ids else column_ids[1] if len(column_ids) > 1 else column_ids[0]

print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

# Filter records on the numeric field (e.g., age > 60)
threshold = 60  # Example threshold
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()

print(f"\nFiltered records where {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field in filtered records
# Ensure dtype is float
filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id and get mean of numeric_field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(6,4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot grouped by group_field_id if available
if group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² clinical dataset using the `mlcroissant` library, referencing all data by their `@id`s for traceability. We reviewed its available record sets, loaded them by `@id`, performed filtering and normalization on a numeric field, grouped by a categorical field, and visualized distributions. 

This framework enables robust and reproducible analysis in accordance with FAIR data principles.